# Sprint 7 - Pair Selection

This notebook demonstrates the research pair-selection flow using deterministic synthetic data. It is not a market result and must not be used as a Sprint 8 gate input until the historical Binance USD-M dataset from `docs/historical_dataset.md` is loaded and versioned.

The production source of truth for behavior is `src/research/pair_selection.py` plus the automated tests.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.research import (
    CorrelationMode,
    ExecutionCostQuality,
    PairSelectionConfig,
    select_pairs,
)

## Synthetic Normalized Bars

The shape below mirrors the normalized fields expected from the historical dataset contract: symbol, UTC open time, research price, complete-bar flag, volume, trade count, funding, and optional execution-spread evidence.

In [ ]:
HOUR_MS = 60 * 60 * 1000
PERIODS = 96
open_time = np.arange(PERIODS, dtype=np.int64) * HOUR_MS
base_returns = np.sin(np.arange(PERIODS - 1) / 4.0) * 0.002 + np.cos(np.arange(PERIODS - 1) / 9.0) * 0.001


def prices_from_returns(start: float, returns: np.ndarray) -> np.ndarray:
    return start * np.exp(np.r_[0.0, np.cumsum(returns)])


def rows_for_symbol(
    symbol: str,
    returns: np.ndarray,
    *,
    start: float,
    quote_volume: float = 5_000_000.0,
    trades: int = 900,
    funding_rate: float = 0.00003,
    execution_cost_quality: str = ExecutionCostQuality.VERIFIED.value,
    median_spread_bps: float = 1.2,
    p95_spread_bps: float = 3.0,
    p99_spread_bps: float = 5.0,
) -> list[dict[str, object]]:
    prices = prices_from_returns(start, returns)
    return [
        {
            "symbol": symbol,
            "open_time": int(ts),
            "price_for_research": float(price),
            "is_complete_bar": True,
            "quote_volume": quote_volume,
            "number_of_trades": trades,
            "funding_rate_asof": funding_rate,
            "execution_cost_quality": execution_cost_quality,
            "median_spread_bps_1h": median_spread_bps,
            "p95_spread_bps_1h": p95_spread_bps,
            "p99_spread_bps_1h": p99_spread_bps,
        }
        for ts, price in zip(open_time, prices, strict=True)
    ]


eth_returns = base_returns * 0.96 + np.sin(np.arange(PERIODS - 1) / 11.0) * 0.0002
sol_returns = np.roll(base_returns, 9) * -0.35 + np.cos(np.arange(PERIODS - 1) / 5.0) * 0.0015

bars = pd.DataFrame(
    rows_for_symbol("BTCUSDT", base_returns, start=60_000.0)
    + rows_for_symbol("ETHUSDT", eth_returns, start=3_000.0)
    + rows_for_symbol("SOLUSDT", sol_returns, start=160.0, execution_cost_quality=ExecutionCostQuality.UNAVAILABLE.value)
    + rows_for_symbol("WEAKUSDT", base_returns, start=1.0, quote_volume=0.0, trades=0, funding_rate=0.0005)
)

bars.head()

## Run Pair Selection

Rolling correlation is shifted one bar by the module, so the score available at a timestamp excludes the current return.

In [ ]:
config = PairSelectionConfig(
    expected_bars=PERIODS,
    min_history_bars=PERIODS,
    min_history_coverage=1.0,
    max_longest_gap_hours=0.0,
    min_funding_coverage=1.0,
    min_reference_price_coverage=1.0,
    min_median_quote_volume=1_000_000.0,
    min_p10_quote_volume=100_000.0,
    min_nonzero_quote_volume_coverage=1.0,
    min_median_trades=100.0,
    max_median_abs_funding_bps=3.0,
    max_p95_abs_funding_bps=15.0,
    max_pair_funding_carry_bps_per_day=10.0,
    min_pair_joint_coverage=1.0,
    min_correlation=0.70,
    correlation_window=24,
    min_correlation_observations=24,
    correlation_mode=CorrelationMode.ROLLING_NO_LOOKAHEAD,
)

selection = select_pairs(bars, config)

In [ ]:
candidate_table = pd.DataFrame(
    [
        {
            "pair": pair.pair_id,
            "score": pair.score,
            "correlation": pair.metrics.correlation,
            "correlation_observations": pair.metrics.correlation_observations,
            "funding_carry_bps_per_day": pair.metrics.funding_carry_bps_per_day,
            "cost_filters_applied": pair.metrics.cost_filters_applied,
            "mode": pair.metrics.correlation_mode,
        }
        for pair in selection.candidate_pairs
    ]
)

candidate_table

In [ ]:
rejected_symbols = pd.DataFrame(
    [
        {
            "symbol": symbol.symbol,
            "reasons": ", ".join(reason.value for reason in symbol.reasons),
            "median_quote_volume": symbol.metrics.median_quote_volume,
            "median_trades": symbol.metrics.median_trades,
            "funding_bps_per_day": symbol.metrics.funding_bps_per_day,
        }
        for symbol in selection.rejected_symbols
    ]
)

rejected_pairs = pd.DataFrame(
    [
        {
            "pair": pair.pair_id,
            "reasons": ", ".join(reason.value for reason in pair.reasons),
            "correlation": pair.metrics.correlation,
            "funding_carry_bps_per_day": pair.metrics.funding_carry_bps_per_day,
        }
        for pair in selection.rejected_pairs
    ]
)

rejected_symbols, rejected_pairs

## Review Notes

- Candidate rows above are synthetic smoke evidence only.
- Real-market approval requires the 36 complete-month Binance USD-M dataset, checksums, and a frozen formation/evaluation policy.
- Execution-cost filters apply only when both legs have verified historical cost evidence; incomplete evidence must fail closed for cost-gated claims.